## This is a process to infer perturbation result by given control cells and perturbed gene name. 

List of candidate gene that can be perturbed in our simulation:

 RIMBP3, HIC2, SNAP29, CRKL, SLC7A4, TMEM191C, CCDC74A, SCARF2, DGCR2, SLC25A1, PI4KA, DGCR6, RTN4R, ZDHHC8, RANBP1, AIFM3, TRMT2A, DGCR8, ARVCF, TXNRD2, GNB1L, LZTR1, MRPL40, MED15, TANGO2, COMT

In [1]:
from ours import preprocess, align_with_train_data, infer

In [1]:
!which python

/home/share/huadjyin/home/s_xiatianyi/.conda/envs/case_sty/bin/python


Describe expression and gene names for your control cells. Format: adata with .X in DENSE array describing UMI counts, and .var_names describing gene names

In [ ]:
# demo to show you the pipeline can work
import anndata
import numpy as np
import os

from pathlib import Path

seed = 42
n = 20
p = 0.5
cells = 50
genes = 5025
samp_times_over_element = 0.9

# zero inflated binomial distribution, to simulate a ctrl expression matrix
np.random.seed(seed)
exp = np.random.binomial(n, p, (cells, genes))
exp[np.random.randint(0, cells-1, (int(cells * genes * samp_times_over_element))), 
    np.random.randint(0, genes-1, (int(cells * genes * samp_times_over_element)))] = 0  # about 60% sparsity as checked

# construct with format as required 
adata = anndata.AnnData(exp)
vnames = np.load(Path(os.getcwd()) / 'infer_by_ctrl/ours/data/var_names.npy')
adata.var_names = vnames

# to simulate unmeasured genes
adata = adata[:, ~adata.var_names.isin(['XKR4', 'SOX17', 'OPRK1'])]

In [3]:
adata

View of AnnData object with n_obs × n_vars = 50 × 5022

Align your data with our trained data, in terms of gene order in adata, as well as preprocessing.

In [3]:
adata = preprocess(adata)
adata_aligned = align_with_train_data(adata)

/home/share/huadjyin/home/s_xiatianyi/.conda/envs/case_sty/lib/python3.9/site-packages/scanpy/preprocessing/_simple.py:137: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata.obs['n_counts'] = number


Unmeasured gene in dataset: 
 ['XKR4' 'SOX17' 'OPRK1']
expression of these genes are set as 0.


Infer expression after perturbation, results is added to adata.layers['case_pred'], with value in scale of data being normalized and log tranformed.

In [5]:
adata_aligned = infer(adata_aligned, pert_gene_name='RIMBP3', batch_size=2, infer_device='cuda:2', label='RIMBP3')
adata_aligned = infer(adata_aligned, pert_gene_name='HIC2', batch_size=2, infer_device='cuda:2', label='HIC2')

RIMBP3 perturbed, labeled as 'RIMBP3'
Infer progress: 100% ▋▋▋▋▋▋▋▋▋▋▋▋▋▋▋▋▋▋▋▋▋▋▋▋▋
HIC2 perturbed, labeled as 'HIC2'
Infer progress: 100% ▋▋▋▋▋▋▋▋▋▋▋▋▋▋▋▋▋▋▋▋▋▋▋▋▋


In [ ]:
adata_aligned